In [15]:
import os
import datetime
import requests
import pandas as pd
from zoneinfo import ZoneInfo

API_KEY = os.environ["SOLAR_EDGE_API_KEY"]
SITE_ID = os.environ["SOLAR_EDGE_SITE_ID"]

BASE_URL = f"https://monitoringapi.solaredge.com/site/{SITE_ID}/powerDetails"

TZ = ZoneInfo("Europe/Berlin")
START_DATE = datetime.datetime(2025, 12, 1, tzinfo=TZ)
END_DATE = datetime.datetime.now(tz=TZ)

METERS = "Production,Consumption,SelfConsumption,FeedIn,Purchased"

# =========================
# Helper: Monatsweise Zeiträume erzeugen
# =========================
def month_ranges(start, end):
    current = start
    while current <= end:
        next_month = (current.replace(day=1) + datetime.timedelta(days=32)).replace(day=1)
        month_end = min(next_month - datetime.timedelta(seconds=1), end)
        yield current, month_end
        current = next_month

# =========================
# API-Abfragen & Records sammeln
# =========================
records = []

for start, end in month_ranges(START_DATE, END_DATE):

    params = {
        "startTime": start.strftime("%Y-%m-%d %H:%M:%S"),
        "endTime": end.strftime("%Y-%m-%d %H:%M:%S"),
        "meters": METERS,
        "api_key": API_KEY
    }

    print(f"Fetching {params['startTime']} → {params['endTime']}")

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    data = response.json()["powerDetails"]

    for meter in data["meters"]:
        meter_type = meter["type"]
        for v in meter["values"]:
            records.append({
                "date": v["date"],
                "type": meter_type,
                "value": v.get("value", 0)
            })

# =========================
# DataFrame bauen
# =========================
df = pd.DataFrame(records)

df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(TZ)

df = (
    df.pivot_table(
        index="date",
        columns="type",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
    .sort_values("date")
)

# =========================
# CSV speichern
# =========================
df.to_csv("energy_data.csv", index=False)

print(df.head())
print(df.tail())

Fetching 2025-12-01 00:00:00 → 2025-12-22 15:50:11
type                      date  Consumption  FeedIn  Production   Purchased  \
0    2025-12-01 00:00:00+01:00     94.64064     0.0         0.0    94.64064   
1    2025-12-01 00:15:00+01:00   1716.24900     0.0         0.0  1716.24900   
2    2025-12-01 00:30:00+01:00   1994.88600     0.0         0.0  1994.88600   
3    2025-12-01 00:45:00+01:00     62.92523     0.0         0.0    62.92523   
4    2025-12-01 01:00:00+01:00    117.45834     0.0         0.0   117.45834   

type  SelfConsumption  
0                 0.0  
1                 0.0  
2                 0.0  
3                 0.0  
4                 0.0  
type                      date  Consumption     FeedIn  Production  Purchased  \
2075 2025-12-22 14:45:00+01:00  1180.495100  265.63156    843.3333   602.7934   
2076 2025-12-22 15:00:00+01:00  1920.533200    0.00000    704.3333  1216.1998   
2077 2025-12-22 15:15:00+01:00    38.368958  537.29770    575.6667     0.0000   
2078 2